# Tratamento dos dados

## Objetivo

Este notebook é a primeira etapa do trabalho. Aqui ainda não vamos comparar modelos de previsão.

O objetivo é obter os dados oficiais, entender como eles estão organizados, definir com clareza quais registros pertencem ao estudo e construir uma base semanal confiável para as etapas posteriores.

O foco do estudo são internações por Síndrome Respiratória Aguda Grave (SRAG) associadas ao vírus sincicial respiratório (VSR) em crianças menores de dois anos, observadas por Unidade da Federação, região e Brasil.

## Etapas previstas

1. Preparar o ambiente de trabalho e organizar as pastas do projeto.
2. Obter os microdados oficiais do SIVEP-Gripe.
3. Inspecionar as colunas, o período disponível e o volume de registros.
4. Definir o recorte do estudo: idade, VSR, local e data de referência.
5. Verificar problemas de qualidade: campos ausentes, duplicidades, datas inválidas e atrasos de notificação.
6. Transformar os registros individuais em séries semanais por UF, região e Brasil.
7. Validar a base final e documentar todas as decisões tomadas.

## Resultado esperado

Ao final, teremos uma única base tratada, reproduzível e documentada. Essa mesma base será usada para avaliar todos os modelos de previsão de forma justa.

In [ ]:
# ETAPA 1 — Preparação do ambiente

from pathlib import Path
from datetime import date
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Organização reprodutível das pastas dentro do ambiente do Colab
PROJETO_DIR = Path("/content/tratamento_vsr")
RAW_DIR = PROJETO_DIR / "dados_brutos"
INTERIM_DIR = PROJETO_DIR / "dados_intermediarios"
PROCESSED_DIR = PROJETO_DIR / "dados_tratados"
OUTPUT_DIR = PROJETO_DIR / "saidas"

for pasta in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    pasta.mkdir(parents=True, exist_ok=True)

# Configurações de visualização
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

print("Ambiente preparado com sucesso.")
print(f"Data de execução: {date.today().isoformat()}")
print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"Diretório do projeto: {PROJETO_DIR}")
print(f"Sistema: {platform.platform()}")

## Etapa 2 — Catálogo oficial dos arquivos SIVEP-Gripe

Os microdados de SRAG são atualizados pelo Ministério da Saúde à medida que as notificações são investigadas e corrigidas.

Nesta etapa, ainda não vamos tratar nem filtrar os registros. Vamos consultar o catálogo oficial do OpenDataSUS e registrar quais arquivos estão disponíveis, seus links e a data em que foram acessados.

Essa etapa é importante porque o recorte temporal e a versão dos arquivos precisam ficar documentados antes de qualquer análise.

In [ ]:
# ETAPA 2 — Consulta reproduzível ao catálogo oficial do OpenDataSUS

import json
from datetime import datetime, timezone

import requests

PORTAL_OPENDATASUS = "https://opendatasus.saude.gov.br"
PACKAGE_ID_SRAG = "39a4995f-4a6e-440f-8c8f-b00c81fae0d0"

url_api = (
    f"{PORTAL_OPENDATASUS}/api/3/action/package_show"
    f"?id={PACKAGE_ID_SRAG}"
)

resposta = requests.get(url_api, timeout=60)
resposta.raise_for_status()

catalogo = resposta.json()

if not catalogo.get("success"):
    raise RuntimeError("A API do OpenDataSUS não retornou sucesso.")

pacote = catalogo["result"]
recursos = pd.DataFrame(pacote["resources"])

# Mantemos somente as informações necessárias para decidir os arquivos
colunas_desejadas = [
    "name", "format", "url", "created", "last_modified", "description"
]
colunas_existentes = [c for c in colunas_desejadas if c in recursos.columns]

recursos_srag = (
    recursos[colunas_existentes]
    .copy()
    .sort_values("name", kind="stable")
    .reset_index(drop=True)
)

momento_consulta_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

manifesto = {
    "fonte": pacote.get("title"),
    "url_catalogo": f"{PORTAL_OPENDATASUS}/dataset/{pacote.get('name', '')}",
    "momento_consulta_utc": momento_consulta_utc,
    "package_id": PACKAGE_ID_SRAG,
    "quantidade_recursos": len(recursos_srag),
}

print("Catálogo consultado com sucesso.")
print(json.dumps(manifesto, ensure_ascii=False, indent=2))

display(recursos_srag)

## Etapa 2A — Catálogo alternativo dos arquivos oficiais

O portal OpenDataSUS pode apresentar instabilidade temporária. Por isso, consultaremos diretamente o repositório oficial de arquivos associado ao portal.

Esta etapa somente lista os arquivos disponíveis por ano. Nenhum dado será baixado, filtrado ou alterado.

In [ ]:
# ETAPA 2A — Listagem alternativa dos arquivos oficiais de SRAG
# Não baixa microdados; apenas consulta o catálogo de arquivos.

from xml.etree import ElementTree as ET
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BUCKET_OFICIAL = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br"
ANOS_ESTUDO = list(range(2019, 2027))

# Tentativas automáticas para instabilidades temporárias de conexão
sessao = requests.Session()

politica_tentativas = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

sessao.mount(
    "https://",
    HTTPAdapter(max_retries=politica_tentativas)
)

def listar_arquivos_srag(ano):
    """Lista os objetos públicos de SRAG disponíveis para um ano."""

    parametros = {
        "list-type": "2",
        "prefix": f"SRAG/{ano}/",
    }

    resposta = sessao.get(
        BUCKET_OFICIAL,
        params=parametros,
        timeout=(20, 90),
    )
    resposta.raise_for_status()

    raiz = ET.fromstring(resposta.content)
    namespace = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

    linhas = []

    for item in raiz.findall("s3:Contents", namespace):
        chave = item.findtext("s3:Key", default="", namespaces=namespace)
        atualizado_em = item.findtext(
            "s3:LastModified",
            default="",
            namespaces=namespace,
        )
        tamanho_bytes = item.findtext(
            "s3:Size",
            default="",
            namespaces=namespace,
        )

        if chave.lower().endswith(".csv"):
            linhas.append(
                {
                    "ano": ano,
                    "arquivo": chave.split("/")[-1],
                    "url": f"{BUCKET_OFICIAL}/{chave}",
                    "atualizado_em_utc": atualizado_em,
                    "tamanho_mb": round(int(tamanho_bytes) / 1024**2, 2),
                }
            )

    return pd.DataFrame(linhas)

catalogos = []
erros_catalogo = []

for ano in ANOS_ESTUDO:
    try:
        catalogo_ano = listar_arquivos_srag(ano)
        catalogos.append(catalogo_ano)
        print(f"{ano}: {len(catalogo_ano)} arquivo(s) CSV encontrado(s).")
    except Exception as erro:
        erros_catalogo.append({"ano": ano, "erro": str(erro)})
        print(f"{ano}: não foi possível consultar. Erro: {erro}")

arquivos_disponiveis = (
    pd.concat(catalogos, ignore_index=True)
    if catalogos
    else pd.DataFrame()
)

print("\nArquivos oficiais encontrados:")
display(arquivos_disponiveis)

if erros_catalogo:
    print("\nAnos com falha na consulta:")
    display(pd.DataFrame(erros_catalogo))

## Etapa 2B — Verificação dos arquivos oficiais fixados

A listagem pública de diretórios do repositório foi bloqueada pelo servidor. Isso não impede o acesso aos arquivos individuais.

Nesta etapa, serão verificadas versões específicas dos arquivos oficiais de SRAG disponibilizados pelo Ministério da Saúde. Antes de baixar qualquer microdado, registraremos a existência, o tamanho, a data de modificação e o identificador de cada arquivo.

A versão será congelada para garantir que todo o estudo possa ser reproduzido posteriormente.

In [ ]:
# ETAPA 2B — Verificação de versões específicas dos arquivos oficiais
# Esta célula NÃO baixa os microdados completos.

import re
from datetime import datetime

S3_OFICIAL = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br"

# Versões oficiais verificadas na captura de 20/10/2026.
# Não representam necessariamente a versão mais recente do portal;
# representam uma versão congelada e reprodutível para este estudo.
ARQUIVOS_OFICIAIS_SRAG = {
    2019: f"{S3_OFICIAL}/SRAG/2019/INFLUD19-23-03-2026.parquet",
    2020: f"{S3_OFICIAL}/SRAG/2020/INFLUD20-23-03-2026.parquet",
    2021: f"{S3_OFICIAL}/SRAG/2021/INFLUD21-23-03-2026.parquet",
    2022: f"{S3_OFICIAL}/SRAG/2022/INFLUD22-23-03-2026.parquet",
    2023: f"{S3_OFICIAL}/SRAG/2023/INFLUD23-23-03-2026.parquet",
    2024: f"{S3_OFICIAL}/SRAG/2024/INFLUD24-23-03-2026.parquet",
    2025: f"{S3_OFICIAL}/SRAG/2025/INFLUD25-19-10-2026.parquet",
    2026: f"{S3_OFICIAL}/SRAG/2026/INFLUD26-19-10-2026.parquet",
}

def data_publicacao_no_nome(url):
    resultado = re.search(r"INFLUD\d{2}-(\d{2})-(\d{2})-(\d{4})", url)
    if resultado is None:
        return None

    dia, mes, ano = resultado.groups()
    return datetime.strptime(
        f"{dia}-{mes}-{ano}",
        "%d-%m-%Y",
    ).date()

def verificar_arquivo(url):
    try:
        resposta = requests.head(
            url,
            allow_redirects=True,
            timeout=(20, 90),
        )

        return {
            "status_http": resposta.status_code,
            "acessivel": resposta.ok,
            "tamanho_mb": round(
                int(resposta.headers.get("Content-Length", 0)) / 1024**2,
                2,
            ),
            "modificado_no_servidor": resposta.headers.get("Last-Modified"),
            "etag": resposta.headers.get("ETag"),
            "erro": None,
        }

    except requests.RequestException as erro:
        return {
            "status_http": None,
            "acessivel": False,
            "tamanho_mb": None,
            "modificado_no_servidor": None,
            "etag": None,
            "erro": str(erro),
        }

linhas_manifesto = []

for ano, url in ARQUIVOS_OFICIAIS_SRAG.items():
    verificacao = verificar_arquivo(url)

    linhas_manifesto.append(
        {
            "ano": ano,
            "arquivo": url.split("/")[-1],
            "data_publicacao_no_nome": data_publicacao_no_nome(url),
            "url_oficial": url,
            **verificacao,
        }
    )

manifesto_arquivos = pd.DataFrame(linhas_manifesto)

display(manifesto_arquivos)

if not manifesto_arquivos["acessivel"].all():
    print("ATENÇÃO: existem arquivos que não puderam ser acessados.")
    display(
        manifesto_arquivos.loc[
            ~manifesto_arquivos["acessivel"]
        ]
    )
else:
    print("Todos os oito arquivos foram verificados com sucesso.")

## Etapa 3 — Download e congelamento dos microdados brutos

Nesta etapa, os arquivos oficiais verificados serão baixados individualmente e preservados sem qualquer alteração.

Para cada arquivo, serão registrados: ano, URL de origem, tamanho esperado, tamanho baixado, data do download e hash SHA-256. O hash permitirá verificar posteriormente se o arquivo usado no estudo é exatamente o mesmo.

In [ ]:
# ETAPA 3 — Download reprodutível dos microdados brutos de SRAG
# Atenção: o processo baixa aproximadamente 334 MB e pode levar alguns minutos.

import hashlib
import os
from datetime import datetime, timezone

from tqdm.auto import tqdm

if not manifesto_arquivos["acessivel"].all():
    raise RuntimeError(
        "Existem arquivos não acessíveis. Não iniciaremos o download parcial."
    )

RAW_SRAG_DIR = RAW_DIR / "SRAG"
RAW_SRAG_DIR.mkdir(parents=True, exist_ok=True)

def calcular_sha256(caminho, tamanho_bloco=1024 * 1024):
    """Calcula o hash SHA-256 de um arquivo já salvo."""
    hash_arquivo = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        while bloco := arquivo.read(tamanho_bloco):
            hash_arquivo.update(bloco)

    return hash_arquivo.hexdigest()

def baixar_arquivo_srag(ano, url, destino):
    """Baixa um arquivo para destino temporário e só o torna final ao concluir."""

    destino.parent.mkdir(parents=True, exist_ok=True)
    destino_temporario = destino.with_suffix(destino.suffix + ".part")

    # Consulta o tamanho oficial antes do download
    resposta_head = requests.head(url, allow_redirects=True, timeout=(20, 90))
    resposta_head.raise_for_status()

    tamanho_esperado = int(resposta_head.headers.get("Content-Length", 0))

    # Se já existir uma cópia completa, ela não é baixada novamente.
    if destino.exists() and destino.stat().st_size == tamanho_esperado:
        return {
            "origem": "arquivo_já_existente",
            "bytes_baixados": destino.stat().st_size,
            "bytes_esperados": tamanho_esperado,
            "sha256": calcular_sha256(destino),
        }

    resposta_download = requests.get(
        url,
        stream=True,
        timeout=(20, 300),
    )
    resposta_download.raise_for_status()

    hash_arquivo = hashlib.sha256()
    bytes_baixados = 0

    with open(destino_temporario, "wb") as arquivo, tqdm(
        total=tamanho_esperado,
        unit="B",
        unit_scale=True,
        desc=f"Baixando {ano}",
    ) as barra:
        for bloco in resposta_download.iter_content(chunk_size=1024 * 1024):
            if not bloco:
                continue

            arquivo.write(bloco)
            hash_arquivo.update(bloco)

            tamanho_bloco = len(bloco)
            bytes_baixados += tamanho_bloco
            barra.update(tamanho_bloco)

    if tamanho_esperado and bytes_baixados != tamanho_esperado:
        raise IOError(
            f"Download incompleto em {ano}: "
            f"{bytes_baixados} bytes baixados; "
            f"{tamanho_esperado} esperados."
        )

    os.replace(destino_temporario, destino)

    return {
        "origem": "download_nesta_execucao",
        "bytes_baixados": bytes_baixados,
        "bytes_esperados": tamanho_esperado,
        "sha256": hash_arquivo.hexdigest(),
    }

registro_downloads = []

for linha in manifesto_arquivos.itertuples(index=False):
    ano = int(linha.ano)
    arquivo = linha.arquivo
    url = linha.url_oficial
    destino = RAW_SRAG_DIR / str(ano) / arquivo

    print(f"\nIniciando {ano}: {arquivo}")

    resultado = baixar_arquivo_srag(
        ano=ano,
        url=url,
        destino=destino,
    )

    registro_downloads.append(
        {
            "ano": ano,
            "arquivo": arquivo,
            "url_oficial": url,
            "caminho_local": str(destino),
            "data_download_utc": datetime.now(timezone.utc).isoformat(),
            **resultado,
        }
    )

manifesto_downloads = pd.DataFrame(registro_downloads)

manifesto_downloads.to_csv(
    RAW_SRAG_DIR / "manifesto_download_srag.csv",
    index=False,
)

display(manifesto_downloads)

print("\nDownload concluído.")
print(
    f"Tamanho total: "
    f"{manifesto_downloads['bytes_baixados'].sum() / 1024**2:.2f} MB"
)

## Etapa 4 — Inspeção estrutural dos microdados

Nesta etapa, serão lidos somente os metadados dos arquivos Parquet, sem carregar os registros individuais na memória.

O objetivo é conhecer a estrutura dos arquivos: quantidade de notificações, quantidade de colunas, nomes dos campos, tipos de dados e diferenças entre os anos.

Ainda não será aplicado nenhum filtro para VSR, idade, local ou hospitalização.

In [ ]:
# ETAPA 4 — Inspeção estrutural, sem ler os registros individuais

import re
from pathlib import Path

import pyarrow.parquet as pq

registros_estrutura = []
campos_por_ano = {}

for linha in manifesto_downloads.itertuples(index=False):
    ano = int(linha.ano)
    caminho = Path(linha.caminho_local)

    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    arquivo_parquet = pq.ParquetFile(caminho)
    metadados = arquivo_parquet.metadata
    esquema = arquivo_parquet.schema_arrow

    campos_por_ano[ano] = {
        campo.name: str(campo.type)
        for campo in esquema
    }

    registros_estrutura.append(
        {
            "ano": ano,
            "arquivo": caminho.name,
            "linhas": metadados.num_rows,
            "colunas": len(esquema),
            "grupos_de_linhas": metadados.num_row_groups,
            "tamanho_mb": round(caminho.stat().st_size / 1024**2, 2),
            "gerado_por": metadados.created_by,
        }
    )

resumo_estrutural = (
    pd.DataFrame(registros_estrutura)
    .sort_values("ano")
    .reset_index(drop=True)
)

# Matriz de presença dos campos em cada ano
todos_os_campos = sorted(
    set().union(*[set(campos.keys()) for campos in campos_por_ano.values()])
)

matriz_campos = pd.DataFrame(
    {
        ano: [campo in campos_por_ano[ano] for campo in todos_os_campos]
        for ano in sorted(campos_por_ano)
    },
    index=todos_os_campos,
)

matriz_campos.index.name = "campo"

campos_em_todos_anos = matriz_campos.index[
    matriz_campos.all(axis=1)
].tolist()

campos_ausentes_por_ano = (
    matriz_campos
    .loc[~matriz_campos.all(axis=1)]
    .reset_index()
)

# Campos potencialmente relevantes para o estudo de saúde
padrao_campos_relevantes = re.compile(
    r"IDADE|DT_|DATA|UF|MUN|HOSP|PCR|VSR|VIRAL|RESULT|EVOL|CLASSI|NOTIF",
    flags=re.IGNORECASE,
)

campos_relevantes = (
    pd.DataFrame(
        {
            "campo": todos_os_campos,
            "presente_em_todos_os_anos": [
                campo in campos_em_todos_anos
                for campo in todos_os_campos
            ],
            "tipo_em_2019": [
                campos_por_ano[2019].get(campo)
                for campo in todos_os_campos
            ],
            "tipo_em_2026": [
                campos_por_ano[2026].get(campo)
                for campo in todos_os_campos
            ],
        }
    )
    .query("campo.str.contains(@padrao_campos_relevantes)", engine="python")
    .sort_values("campo")
    .reset_index(drop=True)
)

# Salvamos o diagnóstico estrutural para auditoria posterior
resumo_estrutural.to_csv(
    OUTPUT_DIR / "resumo_estrutural_arquivos.csv",
    index=False,
)

campos_relevantes.to_csv(
    OUTPUT_DIR / "campos_relevantes_estrutura.csv",
    index=False,
)

print("RESUMO DOS ARQUIVOS")
display(resumo_estrutural)

print(
    f"Campos distintos encontrados: {len(todos_os_campos)}\n"
    f"Campos presentes em todos os anos: {len(campos_em_todos_anos)}\n"
    f"Campos ausentes em pelo menos um ano: "
    f"{len(todos_os_campos) - len(campos_em_todos_anos)}"
)

print("\nCAMPOS POTENCIALMENTE RELEVANTES PARA O ESTUDO")
display(campos_relevantes)

if campos_ausentes_por_ano.empty:
    print("\nA estrutura de campos é idêntica em todos os anos.")
else:
    print("\nExistem diferenças de campos entre anos:")
    display(campos_ausentes_por_ano)

## Etapa 5 — Diagnóstico dos campos epidemiológicos relevantes

Nesta etapa, serão examinados somente campos necessários para construir o recorte do estudo: datas, idade, local de residência, hospitalização e investigação laboratorial.

Ainda não será excluído nenhum registro. O objetivo é entender os códigos utilizados, os valores ausentes e a cobertura dos campos em cada ano.

In [ ]:
# ETAPA 5 — Diagnóstico de campos epidemiológicos, sem filtrar registros

CAMPOS_CANDIDATOS = [
    # Datas
    "DT_NOTIFIC",
    "DT_SIN_PRI",
    "DT_INTERNA",
    "DT_DIGITA",

    # Idade
    "NU_IDADE_N",
    "TP_IDADE",

    # Localização
    "SG_UF_RES",
    "SG_UF_NOT",
    "SG_UF_INTE",

    # Hospitalização e classificação
    "HOSPITAL",
    "CLASSI_FIN",

    # Investigação laboratorial de vírus respiratórios
    "PCR_RESUL",
    "POS_PCROUT",
    "AN_VSR",
    "TP_FLU_PCR",
]

campos_disponiveis = set(todos_os_campos)

campos_selecionados = [
    campo for campo in CAMPOS_CANDIDATOS
    if campo in campos_disponiveis
]

campos_nao_encontrados = [
    campo for campo in CAMPOS_CANDIDATOS
    if campo not in campos_disponiveis
]

print("Campos que serão examinados:")
print(campos_selecionados)

if campos_nao_encontrados:
    print("\nCampos não encontrados no arquivo:")
    print(campos_nao_encontrados)

campos_data = [
    campo for campo in [
        "DT_NOTIFIC",
        "DT_SIN_PRI",
        "DT_INTERNA",
        "DT_DIGITA",
    ]
    if campo in campos_selecionados
]

campos_categoricos = [
    campo for campo in [
        "TP_IDADE",
        "SG_UF_RES",
        "SG_UF_NOT",
        "SG_UF_INTE",
        "HOSPITAL",
        "CLASSI_FIN",
        "PCR_RESUL",
        "POS_PCROUT",
        "AN_VSR",
        "TP_FLU_PCR",
    ]
    if campo in campos_selecionados
]

resumo_qualidade = []
resumo_datas = []
frequencias_categoricas = []

for linha in manifesto_downloads.itertuples(index=False):
    ano = int(linha.ano)
    caminho = Path(linha.caminho_local)

    # Lê somente as colunas selecionadas e processa um ano por vez.
    dados_ano = pd.read_parquet(
        caminho,
        columns=campos_selecionados,
    )

    total_linhas = len(dados_ano)

    for campo in campos_selecionados:
        quantidade_ausente = int(dados_ano[campo].isna().sum())

        resumo_qualidade.append(
            {
                "ano": ano,
                "campo": campo,
                "total_registros": total_linhas,
                "ausentes": quantidade_ausente,
                "percentual_ausente": round(
                    100 * quantidade_ausente / total_linhas,
                    2,
                ),
                "valores_distintos": int(
                    dados_ano[campo].nunique(dropna=True)
                ),
            }
        )

    # Diagnóstico das datas, sem excluir datas inválidas nesta etapa
    for campo in campos_data:
        serie_original = dados_ano[campo]

        serie_data = pd.to_datetime(
            serie_original.astype("string"),
            errors="coerce",
            dayfirst=True,
            format="mixed",
        )

        quantidade_preenchida = int(serie_original.notna().sum())
        quantidade_invalida = int(
            (serie_original.notna() & serie_data.isna()).sum()
        )

        resumo_datas.append(
            {
                "ano": ano,
                "campo_data": campo,
                "preenchidas": quantidade_preenchida,
                "datas_invalidas": quantidade_invalida,
                "data_minima": serie_data.min(),
                "data_maxima": serie_data.max(),
            }
        )

    # Frequências dos códigos: sem interpretação nesta etapa
    for campo in campos_categoricos:
        contagem = (
            dados_ano[campo]
            .astype("string")
            .fillna("<AUSENTE>")
            .value_counts(dropna=False)
            .rename_axis("valor")
            .reset_index(name="quantidade")
        )

        contagem["ano"] = ano
        contagem["campo"] = campo
        frequencias_categoricas.append(contagem)

    del dados_ano

resumo_qualidade = pd.DataFrame(resumo_qualidade)
resumo_datas = pd.DataFrame(resumo_datas)

frequencias_categoricas = pd.concat(
    frequencias_categoricas,
    ignore_index=True,
)

# Não precisamos listar mais de 30 códigos por campo e ano.
frequencias_categoricas = (
    frequencias_categoricas
    .sort_values(
        ["ano", "campo", "quantidade"],
        ascending=[True, True, False],
    )
    .groupby(["ano", "campo"], group_keys=False)
    .head(30)
    .reset_index(drop=True)
)

# Arquivos de auditoria da etapa
resumo_qualidade.to_csv(
    OUTPUT_DIR / "resumo_qualidade_campos.csv",
    index=False,
)

resumo_datas.to_csv(
    OUTPUT_DIR / "resumo_datas_campos.csv",
    index=False,
)

frequencias_categoricas.to_csv(
    OUTPUT_DIR / "frequencias_campos_categoricos.csv",
    index=False,
)

print("\nQUALIDADE DOS CAMPOS")
display(resumo_qualidade)

print("\nCOBERTURA E INTERVALO DAS DATAS")
display(resumo_datas)

print("\nCÓDIGOS E FREQUÊNCIAS DOS CAMPOS CATEGÓRICOS")
display(frequencias_categoricas)

## Etapa 5B — Inventário completo das 194 colunas

Esta etapa cria um inventário técnico de todos os campos disponíveis nos microdados, sem selecionar variáveis por siglas ou por julgamento antecipado.

Para cada campo, serão registrados: presença nos anos, tipo de dado e ausência informada nos metadados do arquivo.

A interpretação epidemiológica será feita na etapa seguinte, com apoio do dicionário oficial do SIVEP-Gripe.

In [ ]:
# ETAPA 5B — Inventário técnico de todas as 194 colunas
# Não lê os registros individuais; usa apenas metadados Parquet.

def obter_nulos_metadados(metadados, indice_coluna):
    """Soma nulos conhecidos nos grupos de linhas do arquivo Parquet."""
    total_nulos = 0

    for indice_grupo in range(metadados.num_row_groups):
        coluna = metadados.row_group(indice_grupo).column(indice_coluna)
        estatisticas = coluna.statistics

        if estatisticas is None or estatisticas.null_count is None:
            return np.nan

        total_nulos += estatisticas.null_count

    return total_nulos

perfil_campos_por_ano = []

for linha in manifesto_downloads.itertuples(index=False):
    ano = int(linha.ano)
    arquivo_parquet = pq.ParquetFile(linha.caminho_local)

    metadados = arquivo_parquet.metadata
    esquema = arquivo_parquet.schema_arrow
    total_linhas = metadados.num_rows

    for indice, campo in enumerate(esquema):
        quantidade_nulos = obter_nulos_metadados(
            metadados,
            indice,
        )

        perfil_campos_por_ano.append(
            {
                "ano": ano,
                "campo": campo.name,
                "tipo_dado": str(campo.type),
                "total_registros": total_linhas,
                "nulos_metadados": quantidade_nulos,
                "percentual_nulos": (
                    round(100 * quantidade_nulos / total_linhas, 4)
                    if pd.notna(quantidade_nulos)
                    else np.nan
                ),
            }
        )

perfil_campos_por_ano = pd.DataFrame(perfil_campos_por_ano)

# Uma linha para cada um dos 194 campos
inventario_194_campos = pd.DataFrame(
    {"campo": sorted(perfil_campos_por_ano["campo"].unique())}
)

for ano in sorted(perfil_campos_por_ano["ano"].unique()):
    perfil_ano = (
        perfil_campos_por_ano
        .query("ano == @ano")
        [["campo", "tipo_dado", "nulos_metadados", "percentual_nulos"]]
        .rename(
            columns={
                "tipo_dado": f"tipo_{ano}",
                "nulos_metadados": f"nulos_{ano}",
                "percentual_nulos": f"percentual_nulos_{ano}",
            }
        )
    )

    inventario_194_campos = inventario_194_campos.merge(
        perfil_ano,
        on="campo",
        how="left",
    )

inventario_194_campos["presente_em_todos_os_anos"] = (
    inventario_194_campos[
        [f"tipo_{ano}" for ano in range(2019, 2027)]
    ]
    .notna()
    .all(axis=1)
)

inventario_194_campos["classificacao_no_estudo"] = "PENDENTE"
inventario_194_campos["decisao_de_uso"] = "PENDENTE"
inventario_194_campos["justificativa"] = "PENDENTE"

perfil_campos_por_ano.to_csv(
    OUTPUT_DIR / "perfil_campos_por_ano.csv",
    index=False,
)

inventario_194_campos.to_csv(
    OUTPUT_DIR / "inventario_194_campos.csv",
    index=False,
)

print(f"Campos inventariados: {len(inventario_194_campos)}")
print(
    "Campos presentes em todos os anos: "
    f"{inventario_194_campos['presente_em_todos_os_anos'].sum()}"
)

display(inventario_194_campos)

## Etapa 6 — Diagnóstico para definição do recorte epidemiológico

Esta etapa investiga, sem excluir registros, os campos necessários para definir o recorte final: tempo de ocorrência, idade, geografia, hospitalização e confirmação laboratorial de VSR.

Os resultados serão salvos em arquivos para análise, sem imprimir tabelas extensas no notebook.

In [ ]:
# ETAPA 6 — Diagnóstico completo para definição do recorte
# Não cria filtros e não exclui registros.

from pathlib import Path
from zipfile import ZipFile

PASTA_ETAPA_6 = OUTPUT_DIR / "etapa_6_definicao_recorte"
PASTA_ETAPA_6.mkdir(parents=True, exist_ok=True)

# Campos selecionados após o inventário das 194 variáveis.
MAPA_CAMPOS = {
    "tempo_ocorrencia": "DT_SIN_PRI",
    "data_notificacao": "DT_NOTIFIC",
    "data_digitacao": "DT_DIGITA",
    "data_internacao": "DT_INTERNA",
    "data_pcr": "DT_PCR",
    "data_resultado_antigeno": "DT_RES_AN",
    "idade_numerica": "NU_IDADE_N",
    "unidade_idade": "TP_IDADE",
    "uf_residencia_candidata": "SG_UF",
    "municipio_residencia": "CO_MUN_RES",
    "uf_notificacao": "SG_UF_NOT",
    "municipio_notificacao": "CO_MUN_NOT",
    "uf_internacao": "SG_UF_INTE",
    "municipio_internacao": "CO_MU_INTE",
    "hospitalizacao": "HOSPITAL",
    "uti": "UTI",
    "evolucao": "EVOLUCAO",
    "classificacao_final": "CLASSI_FIN",
    "resultado_pcr_geral": "PCR_RESUL",
    "vsr_por_pcr": "PCR_VSR",
    "vsr_por_antigeno": "AN_VSR",
    "resultado_antigeno": "RES_AN",
    "outro_virus_por_pcr": "POS_PCROUT",
    "outro_virus_por_antigeno": "POS_AN_OUT",
    "sars_cov_2_por_pcr": "PCR_SARS2",
}

campos_disponiveis = set(inventario_194_campos["campo"])

mapeamento_campos = pd.DataFrame(
    [
        {
            "funcao_no_estudo": funcao,
            "campo": campo,
            "presente_no_arquivo": campo in campos_disponiveis,
            "status": "DIAGNOSTICAR; SEM FILTRO NESTA ETAPA",
        }
        for funcao, campo in MAPA_CAMPOS.items()
    ]
)

campos_ler = mapeamento_campos.loc[
    mapeamento_campos["presente_no_arquivo"],
    "campo",
].tolist()

campos_datas = [
    campo for campo in [
        "DT_SIN_PRI",
        "DT_NOTIFIC",
        "DT_DIGITA",
        "DT_INTERNA",
        "DT_PCR",
        "DT_RES_AN",
    ]
    if campo in campos_ler
]

campos_frequencia = [
    campo for campo in [
        "TP_IDADE",
        "SG_UF",
        "SG_UF_NOT",
        "SG_UF_INTE",
        "HOSPITAL",
        "UTI",
        "EVOLUCAO",
        "CLASSI_FIN",
        "PCR_RESUL",
        "PCR_VSR",
        "AN_VSR",
        "RES_AN",
        "POS_PCROUT",
        "POS_AN_OUT",
        "PCR_SARS2",
    ]
    if campo in campos_ler
]

qualidade = []
datas = []
frequencias = []
geografia = []
idades = []

for linha in manifesto_downloads.itertuples(index=False):
    ano = int(linha.ano)

    dados_ano = pd.read_parquet(
        linha.caminho_local,
        columns=campos_ler,
    )

    total = len(dados_ano)

    # Completude de todos os campos utilizados na definição do recorte
    for campo in campos_ler:
        ausentes = int(dados_ano[campo].isna().sum())

        qualidade.append(
            {
                "ano": ano,
                "campo": campo,
                "total_registros": total,
                "ausentes": ausentes,
                "percentual_ausente": round(100 * ausentes / total, 4),
            }
        )

    # Intervalo e validade técnica das datas
    for campo in campos_datas:
        valores_data = pd.to_datetime(
            dados_ano[campo].astype("string"),
            errors="coerce",
            dayfirst=True,
            format="mixed",
        )

        datas.append(
            {
                "ano": ano,
                "campo": campo,
                "preenchidas": int(dados_ano[campo].notna().sum()),
                "invalidas": int(
                    (dados_ano[campo].notna() & valores_data.isna()).sum()
                ),
                "minima": valores_data.min(),
                "maxima": valores_data.max(),
            }
        )

    # Códigos observados, incluindo ausentes
    for campo in campos_frequencia:
        tabela = (
            dados_ano[campo]
            .astype("string")
            .fillna("<AUSENTE>")
            .value_counts(dropna=False)
            .rename_axis("valor")
            .reset_index(name="quantidade")
        )

        tabela["ano"] = ano
        tabela["campo"] = campo
        tabela["percentual"] = (
            100 * tabela["quantidade"] / total
        ).round(4)

        frequencias.append(tabela)

    # Comparação das três possibilidades geográficas
    campos_geo = ["SG_UF", "SG_UF_NOT", "SG_UF_INTE"]

    for campo_geo in campos_geo:
        if campo_geo in dados_ano.columns:
            geografia.append(
                {
                    "ano": ano,
                    "campo_geografico": campo_geo,
                    "total_registros": total,
                    "ausentes": int(dados_ano[campo_geo].isna().sum()),
                    "percentual_ausente": round(
                        100 * dados_ano[campo_geo].isna().mean(),
                        4,
                    ),
                }
            )

    # Concordância entre residência candidata e os demais locais
    for campo_comparacao in ["SG_UF_NOT", "SG_UF_INTE"]:
        if {"SG_UF", campo_comparacao}.issubset(dados_ano.columns):
            residencia = dados_ano["SG_UF"]
            comparacao = dados_ano[campo_comparacao]
            ambas = residencia.notna() & comparacao.notna()

            geografia.append(
                {
                    "ano": ano,
                    "campo_geografico": f"SG_UF_vs_{campo_comparacao}",
                    "total_registros": total,
                    "ausentes": int((~ambas).sum()),
                    "percentual_ausente": round(
                        100 * (~ambas).mean(),
                        4,
                    ),
                    "mesma_uf_quando_ambas_preenchidas": int(
                        (residencia[ambas] == comparacao[ambas]).sum()
                    ),
                    "ufs_diferentes_quando_ambas_preenchidas": int(
                        (residencia[ambas] != comparacao[ambas]).sum()
                    ),
                }
            )

    # Idade numérica por unidade ainda não interpretada
    if {"NU_IDADE_N", "TP_IDADE"}.issubset(dados_ano.columns):
        idade_numerica = pd.to_numeric(
            dados_ano["NU_IDADE_N"],
            errors="coerce",
        )

        tabela_idade = (
            pd.DataFrame(
                {
                    "unidade_idade": dados_ano["TP_IDADE"]
                    .astype("string")
                    .fillna("<AUSENTE>"),
                    "idade_numerica": idade_numerica,
                }
            )
            .groupby("unidade_idade", dropna=False)["idade_numerica"]
            .agg(
                registros="size",
                idade_minima="min",
                idade_mediana="median",
                idade_maxima="max",
            )
            .reset_index()
        )

        tabela_idade["ano"] = ano
        idades.append(tabela_idade)

    del dados_ano

qualidade = pd.DataFrame(qualidade)
datas = pd.DataFrame(datas)
frequencias = pd.concat(frequencias, ignore_index=True)
geografia = pd.DataFrame(geografia)
idades = pd.concat(idades, ignore_index=True)

arquivos_saida = {
    "mapeamento_campos_recorte.csv": mapeamento_campos,
    "qualidade_campos_recorte.csv": qualidade,
    "diagnostico_datas_recorte.csv": datas,
    "frequencias_codigos_recorte.csv": frequencias,
    "diagnostico_geografia_recorte.csv": geografia,
    "diagnostico_idade_recorte.csv": idades,
}

for nome, tabela in arquivos_saida.items():
    tabela.to_csv(
        PASTA_ETAPA_6 / nome,
        index=False,
        encoding="utf-8-sig",
    )

arquivo_zip = OUTPUT_DIR / "etapa_6_definicao_recorte.zip"

with ZipFile(arquivo_zip, "w") as arquivo:
    for nome in arquivos_saida:
        caminho = PASTA_ETAPA_6 / nome
        arquivo.write(caminho, arcname=nome)

print("Etapa 6 concluída.")
print(f"Pasta dos resultados: {PASTA_ETAPA_6}")
print(f"Arquivo ZIP: {arquivo_zip}")
print(f"Campos analisados: {len(campos_ler)}")
print("Nenhum registro foi filtrado ou excluído.")

## Etapa 7 — Validação das regras de recorte

Nesta etapa, serão verificadas as regras provisórias de idade, hospitalização, geografia e confirmação de VSR.

O objetivo é encontrar contradições entre resultados gerais e específicos de exames, valores inválidos de idade, códigos geográficos inesperados e possíveis duplicidades.

Ainda não será produzida a série semanal final.

In [ ]:
# ETAPA 7 — Validação das regras de recorte, sem criar a base final

from zipfile import ZipFile

PASTA_ETAPA_7 = OUTPUT_DIR / "etapa_7_validacao_regras"
PASTA_ETAPA_7.mkdir(parents=True, exist_ok=True)

UFS_VALIDAS = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
}

CAMPOS_ETAPA_7 = [
    "NU_NOTIFIC",
    "DT_SIN_PRI",
    "NU_IDADE_N",
    "TP_IDADE",
    "SG_UF",
    "HOSPITAL",
    "PCR_RESUL",
    "PCR_VSR",
    "RES_AN",
    "AN_VSR",
]

def normalizar_codigo(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

resumo_validacao = []
consistencia_laboratorial = []
validacao_idade = []
validacao_geografia = []
chaves_candidatas = []

for linha in manifesto_downloads.itertuples(index=False):
    ano = int(linha.ano)

    dados_ano = pd.read_parquet(
        linha.caminho_local,
        columns=CAMPOS_ETAPA_7,
    )

    total = len(dados_ano)

    # Códigos normalizados
    tp_idade = normalizar_codigo(dados_ano["TP_IDADE"])
    hospital = normalizar_codigo(dados_ano["HOSPITAL"])
    uf = normalizar_codigo(dados_ano["SG_UF"])
    pcr_resul = normalizar_codigo(dados_ano["PCR_RESUL"])
    pcr_vsr = normalizar_codigo(dados_ano["PCR_VSR"])
    res_an = normalizar_codigo(dados_ano["RES_AN"])
    an_vsr = normalizar_codigo(dados_ano["AN_VSR"])

    # Idade em meses: 1=dias; 2=meses; 3=anos
    idade_original = pd.to_numeric(
        dados_ano["NU_IDADE_N"],
        errors="coerce",
    )

    idade_meses = pd.Series(np.nan, index=dados_ano.index)

    idade_meses.loc[tp_idade == "1"] = (
        idade_original.loc[tp_idade == "1"] / 30.4375
    )
    idade_meses.loc[tp_idade == "2"] = (
        idade_original.loc[tp_idade == "2"]
    )
    idade_meses.loc[tp_idade == "3"] = (
        idade_original.loc[tp_idade == "3"] * 12
    )

    idade_valida = (
        tp_idade.isin(["1", "2", "3"])
        & idade_original.notna()
        & (idade_original >= 0)
    )

    menor_de_dois = idade_valida & (idade_meses < 24)

    # Critérios provisórios, ainda sem produzir base final
    hospitalizado = hospital == "1"
    uf_valida = uf.isin(UFS_VALIDAS)

    vsr_pcr = pcr_vsr == "1"
    vsr_antigeno = an_vsr == "1"
    vsr_confirmado = vsr_pcr | vsr_antigeno

    # Checagem de coerência entre teste específico e resultado geral
    pcr_vsr_sem_pcr_positivo = vsr_pcr & (pcr_resul != "1")
    an_vsr_sem_antigeno_positivo = vsr_antigeno & (res_an != "1")

    resumo_validacao.append(
        {
            "ano": ano,
            "total_registros": total,
            "idade_valida": int(idade_valida.sum()),
            "idade_invalida": int((~idade_valida).sum()),
            "menores_de_dois": int(menor_de_dois.sum()),
            "hospitalizados_codigo_1": int(hospitalizado.sum()),
            "uf_residencia_valida": int(uf_valida.sum()),
            "vsr_pcr_positivo": int(vsr_pcr.sum()),
            "vsr_antigeno_positivo": int(vsr_antigeno.sum()),
            "vsr_confirmado_uniao": int(vsr_confirmado.sum()),
            "recorte_provisorio_completo": int(
                (
                    menor_de_dois
                    & hospitalizado
                    & uf_valida
                    & vsr_confirmado
                ).sum()
            ),
        }
    )

    consistencia_laboratorial.append(
        {
            "ano": ano,
            "pcr_vsr_positivo_sem_pcr_geral_positivo": int(
                pcr_vsr_sem_pcr_positivo.sum()
            ),
            "antigeno_vsr_positivo_sem_antigeno_geral_positivo": int(
                an_vsr_sem_antigeno_positivo.sum()
            ),
            "vsr_por_pcr_e_antigeno": int(
                (vsr_pcr & vsr_antigeno).sum()
            ),
            "vsr_apenas_pcr": int(
                (vsr_pcr & ~vsr_antigeno).sum()
            ),
            "vsr_apenas_antigeno": int(
                (~vsr_pcr & vsr_antigeno).sum()
            ),
        }
    )

    for codigo in sorted(tp_idade.dropna().unique()):
        mascara = tp_idade == codigo

        validacao_idade.append(
            {
                "ano": ano,
                "codigo_unidade_idade": codigo,
                "registros": int(mascara.sum()),
                "idades_negativas": int(
                    (idade_original[mascara] < 0).sum()
                ),
                "menores_de_dois": int(
                    menor_de_dois[mascara].sum()
                ),
                "idade_minima_meses": idade_meses[mascara].min(),
                "idade_maxima_meses": idade_meses[mascara].max(),
            }
        )

    valores_uf_invalidos = sorted(
        uf[
            uf.notna()
            & ~uf.isin(UFS_VALIDAS)
        ]
        .unique()
        .tolist()
    )

    validacao_geografia.append(
        {
            "ano": ano,
            "uf_valida": int(uf_valida.sum()),
            "uf_ausente": int(uf.isna().sum()),
            "uf_com_codigo_invalido": int(
                (uf.notna() & ~uf.isin(UFS_VALIDAS)).sum()
            ),
            "codigos_invalidos_encontrados": " | ".join(
                valores_uf_invalidos
            ),
        }
    )

    # Chaves apenas do recorte provisório, sem expor identificadores no resultado
    chaves_candidatas.append(
        pd.DataFrame(
            {
                "ano_arquivo": ano,
                "nu_notific": dados_ano.loc[
                    menor_de_dois
                    & hospitalizado
                    & uf_valida
                    & vsr_confirmado,
                    "NU_NOTIFIC",
                ],
            }
        )
    )

    del dados_ano

resumo_validacao = pd.DataFrame(resumo_validacao)
consistencia_laboratorial = pd.DataFrame(consistencia_laboratorial)
validacao_idade = pd.DataFrame(validacao_idade)
validacao_geografia = pd.DataFrame(validacao_geografia)

# Verificação de duplicidade apenas no recorte provisório
chaves_candidatas = pd.concat(
    chaves_candidatas,
    ignore_index=True,
)

chave_valida = (
    chaves_candidatas["nu_notific"]
    .astype("string")
    .str.strip()
)

duplicadas = chave_valida.duplicated(keep=False) & chave_valida.notna()

diagnostico_duplicidade = pd.DataFrame(
    [
        {
            "registros_recorte_provisorio": len(chaves_candidatas),
            "notificacoes_sem_numero": int(chave_valida.isna().sum()),
            "registros_com_numero_duplicado": int(duplicadas.sum()),
            "numeros_distintos_duplicados": int(
                chave_valida[duplicadas].nunique()
            ),
        }
    ]
)

arquivos_saida = {
    "resumo_validacao_regras.csv": resumo_validacao,
    "consistencia_laboratorial_vsr.csv": consistencia_laboratorial,
    "validacao_idade_meses.csv": validacao_idade,
    "validacao_geografia_uf.csv": validacao_geografia,
    "diagnostico_duplicidade_recorte.csv": diagnostico_duplicidade,
}

for nome, tabela in arquivos_saida.items():
    tabela.to_csv(
        PASTA_ETAPA_7 / nome,
        index=False,
        encoding="utf-8-sig",
    )

arquivo_zip = OUTPUT_DIR / "etapa_7_validacao_regras.zip"

with ZipFile(arquivo_zip, "w") as arquivo:
    for nome in arquivos_saida:
        caminho = PASTA_ETAPA_7 / nome
        arquivo.write(caminho, arcname=nome)

print("Etapa 7 concluída.")
print("Nenhuma base tratada foi criada nesta etapa.")
print(f"Pasta de resultados: {PASTA_ETAPA_7}")
print(f"Arquivo ZIP: {arquivo_zip}")

## Etapa 8 — Construção da base individual tratada

Nesta etapa, serão aplicadas as regras de recorte validadas: idade inferior a dois anos, hospitalização, UF de residência válida e confirmação laboratorial de VSR.

A base resultante preservará indicadores do método de diagnóstico de VSR e de possíveis divergências laboratoriais para auditoria posterior.

In [ ]:
# ETAPA 8 — Construção da base individual tratada

from pathlib import Path
from zipfile import ZipFile

PASTA_ETAPA_8 = OUTPUT_DIR / "etapa_8_base_individual"
PASTA_ETAPA_8.mkdir(parents=True, exist_ok=True)

ARQUIVO_BASE_TRATADA = (
    INTERIM_DIR / "casos_vsr_menores2.parquet"
)

UFS_VALIDAS = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
}

CAMPOS_BASE_TRATADA = [
    "NU_NOTIFIC",
    "DT_SIN_PRI",
    "NU_IDADE_N",
    "TP_IDADE",
    "SG_UF",
    "CO_MUN_RES",
    "HOSPITAL",
    "PCR_RESUL",
    "PCR_VSR",
    "RES_AN",
    "AN_VSR",
]

def normalizar_codigo(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .fillna("")
    )

bases_tratadas = []
resumo_tratamento = []

for linha in manifesto_downloads.itertuples(index=False):
    ano = int(linha.ano)

    bruto = pd.read_parquet(
        linha.caminho_local,
        columns=CAMPOS_BASE_TRATADA,
    )

    total_bruto = len(bruto)

    dt_sin_pri = pd.to_datetime(
        bruto["DT_SIN_PRI"],
        errors="coerce",
    )

    idade_original = pd.to_numeric(
        bruto["NU_IDADE_N"],
        errors="coerce",
    )

    tp_idade = normalizar_codigo(bruto["TP_IDADE"])
    hospital = normalizar_codigo(bruto["HOSPITAL"])
    uf_residencia = normalizar_codigo(bruto["SG_UF"])

    pcr_resul = normalizar_codigo(bruto["PCR_RESUL"])
    pcr_vsr = normalizar_codigo(bruto["PCR_VSR"])
    res_an = normalizar_codigo(bruto["RES_AN"])
    an_vsr = normalizar_codigo(bruto["AN_VSR"])

    # Conversão para meses: 1=dias, 2=meses, 3=anos
    idade_meses = pd.Series(np.nan, index=bruto.index, dtype="float64")

    idade_meses.loc[tp_idade == "1"] = (
        idade_original.loc[tp_idade == "1"] / 30.4375
    )
    idade_meses.loc[tp_idade == "2"] = (
        idade_original.loc[tp_idade == "2"]
    )
    idade_meses.loc[tp_idade == "3"] = (
        idade_original.loc[tp_idade == "3"] * 12
    )

    idade_valida = (
        tp_idade.isin(["1", "2", "3"])
        & idade_original.notna()
        & (idade_original >= 0)
    )

    menor_de_dois = idade_valida & (idade_meses < 24)
    hospitalizado = hospital.eq("1")
    uf_valida = uf_residencia.isin(UFS_VALIDAS)

    vsr_pcr = pcr_vsr.eq("1")
    vsr_antigeno = an_vsr.eq("1")
    vsr_confirmado = vsr_pcr | vsr_antigeno

    # Mantemos as divergências como indicadores; não excluímos casos por isso.
    flag_pcr_divergente = vsr_pcr & ~pcr_resul.eq("1")
    flag_antigeno_divergente = vsr_antigeno & ~res_an.eq("1")

    recorte = (
        dt_sin_pri.notna()
        & menor_de_dois
        & hospitalizado
        & uf_valida
        & vsr_confirmado
    )

    tratado = pd.DataFrame(
        {
            "arquivo_ano": ano,
            "nu_notific": bruto["NU_NOTIFIC"].astype("string").str.strip(),
            "dt_sin_pri": dt_sin_pri,
            "idade_meses": idade_meses,
            "tp_idade_original": tp_idade,
            "uf_residencia": uf_residencia,
            "co_mun_res": bruto["CO_MUN_RES"].astype("string"),
            "hospitalizado": hospitalizado,
            "vsr": vsr_confirmado,
            "vsr_pcr": vsr_pcr,
            "vsr_antigeno": vsr_antigeno,
            "metodo_confirmacao_vsr": np.select(
                [
                    vsr_pcr & vsr_antigeno,
                    vsr_pcr,
                    vsr_antigeno,
                ],
                [
                    "PCR_E_ANTIGENO",
                    "PCR",
                    "ANTIGENO",
                ],
                default="NAO_CONFIRMADO",
            ),
            "flag_pcr_divergente": flag_pcr_divergente,
            "flag_antigeno_divergente": flag_antigeno_divergente,
        }
    ).loc[recorte].copy()

    bases_tratadas.append(tratado)

    resumo_tratamento.append(
        {
            "ano": ano,
            "registros_brutos": total_bruto,
            "idade_valida": int(idade_valida.sum()),
            "menores_de_dois": int(menor_de_dois.sum()),
            "hospitalizados": int(hospitalizado.sum()),
            "uf_residencia_valida": int(uf_valida.sum()),
            "vsr_confirmado": int(vsr_confirmado.sum()),
            "registros_apos_recorte_antes_deduplicacao": len(tratado),
        }
    )

    del bruto, tratado

casos_tratados = pd.concat(
    bases_tratadas,
    ignore_index=True,
)

# Verificação de duplicidade sem expor identificadores nos arquivos de resultado
chave_valida = (
    casos_tratados["nu_notific"].notna()
    & casos_tratados["nu_notific"].ne("")
)

duplicados = (
    chave_valida
    & casos_tratados["nu_notific"].duplicated(keep=False)
)

colunas_criticas = [
    "dt_sin_pri",
    "idade_meses",
    "uf_residencia",
    "vsr_pcr",
    "vsr_antigeno",
]

conflitos_duplicidade = 0

for _, grupo in casos_tratados.loc[duplicados].groupby("nu_notific"):
    possui_conflito = any(
        grupo[coluna].nunique(dropna=False) > 1
        for coluna in colunas_criticas
    )

    conflitos_duplicidade += int(possui_conflito)

if conflitos_duplicidade > 0:
    raise RuntimeError(
        "Foram encontradas notificações duplicadas com informações "
        "críticas conflitantes. A base não foi gerada."
    )

casos_tratados = casos_tratados.drop_duplicates(
    subset=["nu_notific"],
    keep="first",
).reset_index(drop=True)

# Salva a base tratada individual
casos_tratados.to_parquet(
    ARQUIVO_BASE_TRATADA,
    index=False,
)

resumo_tratamento = pd.DataFrame(resumo_tratamento)

diagnostico_duplicidade_final = pd.DataFrame(
    [
        {
            "registros_antes_deduplicacao": int(
                sum(
                    resumo_tratamento[
                        "registros_apos_recorte_antes_deduplicacao"
                    ]
                )
            ),
            "registros_apos_deduplicacao": len(casos_tratados),
            "registros_removidos_por_duplicidade": int(duplicados.sum()),
            "chaves_com_conflito_critico": conflitos_duplicidade,
        }
    ]
)

auditoria_metodo_vsr = (
    casos_tratados
    .groupby(
        ["arquivo_ano", "metodo_confirmacao_vsr"],
        dropna=False,
    )
    .size()
    .reset_index(name="registros")
)

resumo_tratamento.to_csv(
    PASTA_ETAPA_8 / "resumo_tratamento_base_individual.csv",
    index=False,
    encoding="utf-8-sig",
)

diagnostico_duplicidade_final.to_csv(
    PASTA_ETAPA_8 / "diagnostico_duplicidade_final.csv",
    index=False,
    encoding="utf-8-sig",
)

auditoria_metodo_vsr.to_csv(
    PASTA_ETAPA_8 / "auditoria_metodo_confirmacao_vsr.csv",
    index=False,
    encoding="utf-8-sig",
)

arquivo_zip = OUTPUT_DIR / "etapa_8_base_individual_auditoria.zip"

with ZipFile(arquivo_zip, "w") as arquivo:
    for nome in [
        "resumo_tratamento_base_individual.csv",
        "diagnostico_duplicidade_final.csv",
        "auditoria_metodo_confirmacao_vsr.csv",
    ]:
        caminho = PASTA_ETAPA_8 / nome
        arquivo.write(caminho, arcname=nome)

print("Etapa 8 concluída.")
print(f"Base individual tratada: {ARQUIVO_BASE_TRATADA}")
print(f"Registros finais: {len(casos_tratados):,}")
print(f"Resultados de auditoria: {PASTA_ETAPA_8}")
print(f"Arquivo ZIP de auditoria: {arquivo_zip}")

## Etapa 8A — Investigação de duplicidades conflitantes

Foram encontradas notificações duplicadas com diferenças em campos importantes.

Esta etapa não altera a base. Ela produz um relatório agregado, sem expor números de notificação, para identificar quais campos estão divergindo e decidir uma regra de deduplicação justificável.

In [ ]:
# ETAPA 8A — Diagnóstico de notificações duplicadas conflitantes
# Esta célula usa a variável casos_tratados criada pela Etapa 8.

if "casos_tratados" not in globals():
    raise RuntimeError(
        "A variável casos_tratados não está disponível. "
        "Execute novamente a Etapa 8 até ocorrer o erro e rode esta célula."
    )

PASTA_ETAPA_8A = OUTPUT_DIR / "etapa_8a_duplicidades"
PASTA_ETAPA_8A.mkdir(parents=True, exist_ok=True)

chave_valida = (
    casos_tratados["nu_notific"].notna()
    & casos_tratados["nu_notific"].ne("")
)

duplicados = (
    chave_valida
    & casos_tratados["nu_notific"].duplicated(keep=False)
)

colunas_criticas = [
    "arquivo_ano",
    "dt_sin_pri",
    "idade_meses",
    "uf_residencia",
    "vsr_pcr",
    "vsr_antigeno",
    "metodo_confirmacao_vsr",
    "flag_pcr_divergente",
    "flag_antigeno_divergente",
]

grupos_duplicados = []
conflitos_por_campo = []

for _, grupo in casos_tratados.loc[duplicados].groupby("nu_notific"):
    campos_conflitantes = []

    for campo in colunas_criticas:
        valores_distintos = grupo[campo].nunique(dropna=False)

        if valores_distintos > 1:
            campos_conflitantes.append(campo)

            conflitos_por_campo.append(
                {
                    "campo": campo,
                    "registros_no_grupo": len(grupo),
                    "valores_distintos": valores_distintos,
                }
            )

    grupos_duplicados.append(
        {
            "quantidade_registros_no_grupo": len(grupo),
            "anos_arquivo_envolvidos": " | ".join(
                map(
                    str,
                    sorted(grupo["arquivo_ano"].unique()),
                )
            ),
            "quantidade_campos_conflitantes": len(
                campos_conflitantes
            ),
            "campos_conflitantes": " | ".join(
                campos_conflitantes
            ),
        }
    )

grupos_duplicados = pd.DataFrame(grupos_duplicados)

conflitos_por_campo = (
    pd.DataFrame(conflitos_por_campo)
    .groupby("campo", as_index=False)
    .agg(
        grupos_com_conflito=("campo", "size"),
        maior_numero_de_valores_distintos=(
            "valores_distintos",
            "max",
        ),
    )
    .sort_values(
        "grupos_com_conflito",
        ascending=False,
    )
)

resumo_duplicidades = pd.DataFrame(
    [
        {
            "registros_no_recorte_provisorio": len(casos_tratados),
            "registros_duplicados": int(duplicados.sum()),
            "numeros_de_notificacao_duplicados": int(
                casos_tratados.loc[
                    duplicados,
                    "nu_notific",
                ].nunique()
            ),
            "grupos_com_algum_conflito": int(
                (
                    grupos_duplicados[
                        "quantidade_campos_conflitantes"
                    ] > 0
                ).sum()
            ),
        }
    ]
)

resumo_duplicidades.to_csv(
    PASTA_ETAPA_8A / "resumo_duplicidades_conflitantes.csv",
    index=False,
    encoding="utf-8-sig",
)

grupos_duplicados.to_csv(
    PASTA_ETAPA_8A / "grupos_duplicados_sem_identificador.csv",
    index=False,
    encoding="utf-8-sig",
)

conflitos_por_campo.to_csv(
    PASTA_ETAPA_8A / "conflitos_duplicidade_por_campo.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Diagnóstico de duplicidades concluído.")
print(f"Pasta de resultados: {PASTA_ETAPA_8A}")
print(f"Registros duplicados: {int(duplicados.sum())}")
print(
    "Números de notificação duplicados: "
    f"{int(casos_tratados.loc[duplicados, 'nu_notific'].nunique())}"
)

## Etapa 8B — Investigação temporal das duplicidades

Existem duas notificações duplicadas com datas de início de sintomas diferentes.

Nesta etapa, vamos comparar as versões disponíveis de cada notificação usando datas administrativas e laboratoriais, sem expor os identificadores das pessoas.

In [ ]:
# ETAPA 8B — Detalhamento temporal das duplicidades sem identificadores

if "casos_tratados" not in globals():
    raise RuntimeError(
        "A variável casos_tratados não está disponível. "
        "Execute a Etapa 8 novamente até o erro e rode a Etapa 8A."
    )

PASTA_ETAPA_8B = OUTPUT_DIR / "etapa_8b_investigacao_duplicidades"
PASTA_ETAPA_8B.mkdir(parents=True, exist_ok=True)

chave_valida = (
    casos_tratados["nu_notific"].notna()
    & casos_tratados["nu_notific"].ne("")
)

mascara_duplicados = (
    chave_valida
    & casos_tratados["nu_notific"].duplicated(keep=False)
)

ids_duplicados = sorted(
    casos_tratados.loc[
        mascara_duplicados,
        "nu_notific",
    ]
    .astype("string")
    .unique()
    .tolist()
)

mapa_grupos = {
    identificador: f"GRUPO_{indice:02d}"
    for indice, identificador in enumerate(
        ids_duplicados,
        start=1,
    )
}

CAMPOS_INVESTIGACAO = [
    "NU_NOTIFIC",
    "DT_SIN_PRI",
    "DT_NOTIFIC",
    "DT_DIGITA",
    "DT_ENCERRA",
    "DT_PCR",
    "DT_RES_AN",
    "PCR_RESUL",
    "PCR_VSR",
    "RES_AN",
    "AN_VSR",
    "HOSPITAL",
    "SG_UF",
    "NU_IDADE_N",
    "TP_IDADE",
]

detalhes = []

for linha in manifesto_downloads.itertuples(index=False):
    ano_arquivo = int(linha.ano)

    bruto = pd.read_parquet(
        linha.caminho_local,
        columns=CAMPOS_INVESTIGACAO,
    )

    bruto["NU_NOTIFIC"] = (
        bruto["NU_NOTIFIC"]
        .astype("string")
        .str.strip()
    )

    selecao = bruto[
        bruto["NU_NOTIFIC"].isin(ids_duplicados)
    ].copy()

    if selecao.empty:
        continue

    selecao["grupo_anonimo"] = (
        selecao["NU_NOTIFIC"].map(mapa_grupos)
    )

    selecao["arquivo_ano"] = ano_arquivo

    for campo_data in [
        "DT_SIN_PRI",
        "DT_NOTIFIC",
        "DT_DIGITA",
        "DT_ENCERRA",
        "DT_PCR",
        "DT_RES_AN",
    ]:
        selecao[campo_data] = pd.to_datetime(
            selecao[campo_data],
            errors="coerce",
        )

    detalhes.append(
        selecao[
            [
                "grupo_anonimo",
                "arquivo_ano",
                "DT_SIN_PRI",
                "DT_NOTIFIC",
                "DT_DIGITA",
                "DT_ENCERRA",
                "DT_PCR",
                "DT_RES_AN",
                "PCR_RESUL",
                "PCR_VSR",
                "RES_AN",
                "AN_VSR",
                "HOSPITAL",
                "SG_UF",
                "NU_IDADE_N",
                "TP_IDADE",
            ]
        ]
    )

    del bruto, selecao

detalhes_duplicidades = (
    pd.concat(detalhes, ignore_index=True)
    .sort_values(
        ["grupo_anonimo", "DT_DIGITA", "arquivo_ano"],
        na_position="first",
    )
    .reset_index(drop=True)
)

resumo_por_grupo = (
    detalhes_duplicidades
    .groupby("grupo_anonimo", as_index=False)
    .agg(
        versoes_encontradas=("arquivo_ano", "size"),
        anos_arquivo=(
            "arquivo_ano",
            lambda x: " | ".join(map(str, sorted(x.unique()))),
        ),
        menor_data_sintoma=("DT_SIN_PRI", "min"),
        maior_data_sintoma=("DT_SIN_PRI", "max"),
        menor_data_digitacao=("DT_DIGITA", "min"),
        maior_data_digitacao=("DT_DIGITA", "max"),
        menor_data_notificacao=("DT_NOTIFIC", "min"),
        maior_data_notificacao=("DT_NOTIFIC", "max"),
    )
)

detalhes_duplicidades.to_csv(
    PASTA_ETAPA_8B / "detalhe_duplicidades_sem_identificador.csv",
    index=False,
    encoding="utf-8-sig",
)

resumo_por_grupo.to_csv(
    PASTA_ETAPA_8B / "resumo_temporal_duplicidades.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Etapa 8B concluída.")
print(f"Pasta dos resultados: {PASTA_ETAPA_8B}")
print(
    "Grupos de notificações investigados: "
    f"{detalhes_duplicidades['grupo_anonimo'].nunique()}"
)
print(
    "Versões de registros analisadas: "
    f"{len(detalhes_duplicidades)}"
)

In [ ]:
# ETAPA 8C — Deduplicação temporal e geração da base individual final

if "casos_tratados" not in globals():
    raise RuntimeError(
        "A variável casos_tratados não está disponível. "
        "Execute novamente a Etapa 8, a Etapa 8A e a Etapa 8B."
    )

PASTA_ETAPA_8C = OUTPUT_DIR / "etapa_8c_base_final"
PASTA_ETAPA_8C.mkdir(parents=True, exist_ok=True)

ARQUIVO_BASE_FINAL = (
    INTERIM_DIR / "casos_vsr_menores2.parquet"
)

# Identifica as chaves duplicadas na base provisória
chave_valida = (
    casos_tratados["nu_notific"].notna()
    & casos_tratados["nu_notific"].ne("")
)

mascara_duplicados = (
    chave_valida
    & casos_tratados["nu_notific"].duplicated(keep=False)
)

ids_duplicados = casos_tratados.loc[
    mascara_duplicados,
    "nu_notific",
].astype("string").unique().tolist()

# Releitura apenas das quatro versões duplicadas, com datas administrativas
campos_releitura = [
    "NU_NOTIFIC",
    "DT_SIN_PRI",
    "DT_NOTIFIC",
    "DT_DIGITA",
]

versoes_duplicadas = []

for linha in manifesto_downloads.itertuples(index=False):
    bruto = pd.read_parquet(
        linha.caminho_local,
        columns=campos_releitura,
    )

    bruto["NU_NOTIFIC"] = (
        bruto["NU_NOTIFIC"]
        .astype("string")
        .str.strip()
    )

    selecao = bruto[
        bruto["NU_NOTIFIC"].isin(ids_duplicados)
    ].copy()

    if not selecao.empty:
        selecao["arquivo_ano"] = int(linha.ano)

        for campo_data in [
            "DT_SIN_PRI",
            "DT_NOTIFIC",
            "DT_DIGITA",
        ]:
            selecao[campo_data] = pd.to_datetime(
                selecao[campo_data],
                errors="coerce",
            )

        versoes_duplicadas.append(selecao)

    del bruto, selecao

versoes_duplicadas = pd.concat(
    versoes_duplicadas,
    ignore_index=True,
)

# Diferença entre sintomas e notificação
versoes_duplicadas["lag_notificacao_dias"] = (
    versoes_duplicadas["DT_NOTIFIC"]
    - versoes_duplicadas["DT_SIN_PRI"]
).dt.days

versoes_duplicadas["lag_valido"] = (
    versoes_duplicadas["lag_notificacao_dias"].notna()
    & (versoes_duplicadas["lag_notificacao_dias"] >= 0)
)

# Escolhe a versão mais coerente temporalmente
versoes_duplicadas["_ord_lag"] = np.where(
    versoes_duplicadas["lag_valido"],
    versoes_duplicadas["lag_notificacao_dias"],
    np.inf,
)

versoes_duplicadas = versoes_duplicadas.sort_values(
    [
        "NU_NOTIFIC",
        "lag_valido",
        "_ord_lag",
        "DT_DIGITA",
        "arquivo_ano",
    ],
    ascending=[
        True,
        False,
        True,
        False,
        False,
    ],
)

versoes_escolhidas = (
    versoes_duplicadas
    .groupby("NU_NOTIFIC", as_index=False)
    .first()
)

# Mantém todas as não duplicadas e apenas a versão escolhida das duplicadas
casos_tratados["_manter"] = ~mascara_duplicados

for escolha in versoes_escolhidas.itertuples(index=False):
    mascara_escolhida = (
        casos_tratados["nu_notific"].eq(escolha.NU_NOTIFIC)
        & casos_tratados["arquivo_ano"].eq(escolha.arquivo_ano)
        & casos_tratados["dt_sin_pri"].eq(escolha.DT_SIN_PRI)
    )

    casos_tratados.loc[mascara_escolhida, "_manter"] = True

casos_finais = (
    casos_tratados
    .loc[casos_tratados["_manter"]]
    .drop(columns="_manter")
    .copy()
    .reset_index(drop=True)
)

# A base final deve ter uma única linha por notificação
if casos_finais["nu_notific"].duplicated().any():
    raise RuntimeError(
        "A deduplicação não eliminou todas as notificações repetidas."
    )

casos_finais.to_parquet(
    ARQUIVO_BASE_FINAL,
    index=False,
)

# Relatório anonimizado da regra aplicada
mapa_grupos = {
    identificador: f"GRUPO_{indice:02d}"
    for indice, identificador in enumerate(
        sorted(ids_duplicados),
        start=1,
    )
}

auditoria_escolhas = versoes_escolhidas.assign(
    grupo_anonimo=lambda tabela: tabela["NU_NOTIFIC"].map(mapa_grupos),
)[
    [
        "grupo_anonimo",
        "arquivo_ano",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
        "DT_DIGITA",
        "lag_notificacao_dias",
        "lag_valido",
    ]
].rename(
    columns={
        "arquivo_ano": "arquivo_ano_selecionado",
        "DT_SIN_PRI": "data_sintoma_selecionada",
        "DT_NOTIFIC": "data_notificacao",
        "DT_DIGITA": "data_digitacao",
    }
)

resumo_base_final = pd.DataFrame(
    [
        {
            "registros_antes_deduplicacao": len(casos_tratados),
            "registros_removidos": len(casos_tratados) - len(casos_finais),
            "registros_finais": len(casos_finais),
            "notificacoes_duplicadas_restantes": int(
                casos_finais["nu_notific"].duplicated().sum()
            ),
        }
    ]
)

auditoria_metodo_vsr_final = (
    casos_finais
    .groupby(
        ["arquivo_ano", "metodo_confirmacao_vsr"],
        dropna=False,
    )
    .size()
    .reset_index(name="registros")
)

auditoria_escolhas.to_csv(
    PASTA_ETAPA_8C / "auditoria_escolhas_duplicidade.csv",
    index=False,
    encoding="utf-8-sig",
)

resumo_base_final.to_csv(
    PASTA_ETAPA_8C / "resumo_base_individual_final.csv",
    index=False,
    encoding="utf-8-sig",
)

auditoria_metodo_vsr_final.to_csv(
    PASTA_ETAPA_8C / "auditoria_metodo_confirmacao_final.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Etapa 8C concluída.")
print(f"Base individual final: {ARQUIVO_BASE_FINAL}")
print(f"Registros finais: {len(casos_finais):,}")
print(f"Auditorias: {PASTA_ETAPA_8C}")

## Etapa 9 — Construção do painel semanal

Nesta etapa, a base individual tratada será agregada por semana epidemiológica, UF de residência, região e Brasil.

A última semana disponível será marcada como incompleta, pois o arquivo ainda pode receber notificações e correções. Ela não será interpretada como ausência de casos.

In [ ]:
# ETAPA 9 — Painel semanal por UF, região e Brasil

from pathlib import Path
from zipfile import ZipFile

PASTA_ETAPA_9 = OUTPUT_DIR / "etapa_9_painel_semanal"
PASTA_ETAPA_9.mkdir(parents=True, exist_ok=True)

ARQUIVO_BASE_FINAL = (
    INTERIM_DIR / "casos_vsr_menores2.parquet"
)

ARQUIVO_PAINEL_SEMANAL = (
    PROCESSED_DIR / "painel_vsr_semanal.parquet"
)

REGIAO_POR_UF = {
    "AC": "NORTE", "AP": "NORTE", "AM": "NORTE",
    "PA": "NORTE", "RO": "NORTE", "RR": "NORTE", "TO": "NORTE",
    "AL": "NORDESTE", "BA": "NORDESTE", "CE": "NORDESTE",
    "MA": "NORDESTE", "PB": "NORDESTE", "PE": "NORDESTE",
    "PI": "NORDESTE", "RN": "NORDESTE", "SE": "NORDESTE",
    "DF": "CENTRO_OESTE", "GO": "CENTRO_OESTE",
    "MS": "CENTRO_OESTE", "MT": "CENTRO_OESTE",
    "ES": "SUDESTE", "MG": "SUDESTE", "RJ": "SUDESTE", "SP": "SUDESTE",
    "PR": "SUL", "RS": "SUL", "SC": "SUL",
}

ORDENACAO_UFS = sorted(REGIAO_POR_UF.keys())
ORDENACAO_REGIOES = [
    "NORTE",
    "NORDESTE",
    "CENTRO_OESTE",
    "SUDESTE",
    "SUL",
]

if not ARQUIVO_BASE_FINAL.exists():
    raise FileNotFoundError(
        f"Base individual não encontrada: {ARQUIVO_BASE_FINAL}"
    )

casos = pd.read_parquet(ARQUIVO_BASE_FINAL)

casos["dt_sin_pri"] = pd.to_datetime(
    casos["dt_sin_pri"],
    errors="coerce",
)

if casos["dt_sin_pri"].isna().any():
    raise RuntimeError(
        "A base final possui datas de sintomas inválidas."
    )

casos["regiao"] = casos["uf_residencia"].map(REGIAO_POR_UF)

if casos["regiao"].isna().any():
    raise RuntimeError(
        "Foram encontradas UFs sem região correspondente."
    )

# Domingo de início da semana epidemiológica
casos["semana"] = (
    casos["dt_sin_pri"].dt.normalize()
    - pd.to_timedelta(
        (casos["dt_sin_pri"].dt.dayofweek + 1) % 7,
        unit="D",
    )
)

# Ano e semana epidemiológica: a quinta-feira determina o ano epidemiológico
quinta_feira = casos["semana"] + pd.Timedelta(days=4)
calendario_epi = quinta_feira.dt.isocalendar()

casos["ano_epi"] = calendario_epi.year.astype(int)
casos["semana_epi"] = calendario_epi.week.astype(int)

casos["casos_vsr"] = 1
casos["casos_vsr_pcr"] = casos["vsr_pcr"].astype(int)
casos["casos_vsr_antigeno"] = casos["vsr_antigeno"].astype(int)

COLUNAS_CONTAGEM = [
    "casos_vsr",
    "casos_vsr_pcr",
    "casos_vsr_antigeno",
]

# Grade completa de semanas e UFs
primeira_semana = casos["semana"].min()
ultima_semana = casos["semana"].max()

todas_as_semanas = pd.date_range(
    start=primeira_semana,
    end=ultima_semana,
    freq="7D",
)

grade_uf = pd.MultiIndex.from_product(
    [ORDENACAO_UFS, todas_as_semanas],
    names=["unidade", "semana"],
).to_frame(index=False)

agregado_uf = (
    casos
    .groupby(["uf_residencia", "semana"], as_index=False)
    [COLUNAS_CONTAGEM]
    .sum()
    .rename(columns={"uf_residencia": "unidade"})
)

painel_uf = grade_uf.merge(
    agregado_uf,
    on=["unidade", "semana"],
    how="left",
)

painel_uf[COLUNAS_CONTAGEM] = (
    painel_uf[COLUNAS_CONTAGEM]
    .fillna(0)
    .astype("float64")
)

painel_uf["nivel"] = "UF"
painel_uf["regiao"] = painel_uf["unidade"].map(REGIAO_POR_UF)

# A última semana tem cobertura parcial do arquivo; não usar zero como informação.
painel_uf["cobertura_fechada"] = (
    painel_uf["semana"] < ultima_semana
)

painel_uf["semanas_ate_ultima_observada"] = (
    (ultima_semana - painel_uf["semana"]).dt.days // 7
).astype(int)

painel_uf.loc[
    ~painel_uf["cobertura_fechada"],
    COLUNAS_CONTAGEM,
] = np.nan

# Agregação regional a partir das UFs
painel_regiao = (
    painel_uf
    .groupby(["regiao", "semana"], as_index=False)
    [COLUNAS_CONTAGEM]
    .sum(min_count=1)
    .rename(columns={"regiao": "unidade"})
)

painel_regiao["nivel"] = "REGIAO"
painel_regiao["regiao"] = painel_regiao["unidade"]
painel_regiao["cobertura_fechada"] = (
    painel_regiao["semana"] < ultima_semana
)
painel_regiao["semanas_ate_ultima_observada"] = (
    (ultima_semana - painel_regiao["semana"]).dt.days // 7
).astype(int)

# Agregação do Brasil a partir das UFs
painel_brasil = (
    painel_uf
    .groupby("semana", as_index=False)
    [COLUNAS_CONTAGEM]
    .sum(min_count=1)
)

painel_brasil["unidade"] = "BRASIL"
painel_brasil["nivel"] = "BRASIL"
painel_brasil["regiao"] = "BRASIL"
painel_brasil["cobertura_fechada"] = (
    painel_brasil["semana"] < ultima_semana
)
painel_brasil["semanas_ate_ultima_observada"] = (
    (ultima_semana - painel_brasil["semana"]).dt.days // 7
).astype(int)

painel_semanal = pd.concat(
    [painel_uf, painel_regiao, painel_brasil],
    ignore_index=True,
)

# Ano e semana epidemiológica no painel completo
quinta_painel = painel_semanal["semana"] + pd.Timedelta(days=4)
calendario_painel = quinta_painel.dt.isocalendar()

painel_semanal["ano_epi"] = calendario_painel.year.astype(int)
painel_semanal["semana_epi"] = calendario_painel.week.astype(int)

painel_semanal = painel_semanal[
    [
        "nivel",
        "unidade",
        "regiao",
        "semana",
        "ano_epi",
        "semana_epi",
        "casos_vsr",
        "casos_vsr_pcr",
        "casos_vsr_antigeno",
        "cobertura_fechada",
        "semanas_ate_ultima_observada",
    ]
].sort_values(
    ["nivel", "unidade", "semana"]
).reset_index(drop=True)

# Validação: Brasil deve ser igual à soma das UFs em semanas fechadas
uf_fechada = painel_semanal[
    (painel_semanal["nivel"] == "UF")
    & (painel_semanal["cobertura_fechada"])
]

brasil_fechado = painel_semanal[
    (painel_semanal["nivel"] == "BRASIL")
    & (painel_semanal["cobertura_fechada"])
].set_index("semana")

soma_uf = (
    uf_fechada
    .groupby("semana")["casos_vsr"]
    .sum()
)

coerente_brasil = np.allclose(
    soma_uf.loc[brasil_fechado.index].values,
    brasil_fechado["casos_vsr"].values,
)

if not coerente_brasil:
    raise RuntimeError(
        "A soma das UFs não coincide com o total do Brasil."
    )

painel_semanal.to_parquet(
    ARQUIVO_PAINEL_SEMANAL,
    index=False,
)

resumo_painel = (
    painel_semanal
    .groupby("nivel", as_index=False)
    .agg(
        unidades=("unidade", "nunique"),
        semanas=("semana", "nunique"),
        linhas=("nivel", "size"),
        semanas_fechadas=(
            "cobertura_fechada",
            "sum",
        ),
        casos_vsr_observados=(
            "casos_vsr",
            "sum",
        ),
    )
)

validacao_hierarquia = pd.DataFrame(
    [
        {
            "soma_ufs_igual_brasil_em_semanas_fechadas": coerente_brasil,
            "primeira_semana": primeira_semana,
            "ultima_semana_parcial": ultima_semana,
            "quantidade_semanas": len(todas_as_semanas),
            "unidades_uf": len(ORDENACAO_UFS),
            "unidades_regiao": len(ORDENACAO_REGIOES),
        }
    ]
)

resumo_painel.to_csv(
    PASTA_ETAPA_9 / "resumo_painel_semanal.csv",
    index=False,
    encoding="utf-8-sig",
)

validacao_hierarquia.to_csv(
    PASTA_ETAPA_9 / "validacao_hierarquia_painel.csv",
    index=False,
    encoding="utf-8-sig",
)

arquivo_zip = OUTPUT_DIR / "etapa_9_painel_semanal_auditoria.zip"

with ZipFile(arquivo_zip, "w") as arquivo:
    for nome in [
        "resumo_painel_semanal.csv",
        "validacao_hierarquia_painel.csv",
    ]:
        caminho = PASTA_ETAPA_9 / nome
        arquivo.write(caminho, arcname=nome)

print("Etapa 9 concluída.")
print(f"Painel semanal: {ARQUIVO_PAINEL_SEMANAL}")
print(f"Primeira semana: {primeira_semana.date()}")
print(f"Última semana parcial: {ultima_semana.date()}")
print(f"Quantidade de semanas: {len(todas_as_semanas)}")
print(f"Linhas no painel: {len(painel_semanal):,}")
print(f"Auditorias: {PASTA_ETAPA_9}")

## Etapa 10 — Diagnóstico de atrasos de disponibilidade

Nesta etapa, será estimado quanto tempo leva para um caso com início de sintomas aparecer de forma utilizável na base de vigilância.

A disponibilidade do caso será definida pela data mais tardia entre o registro no sistema e a confirmação laboratorial de VSR. Esses atrasos serão usados posteriormente para tratar semanas recentes de forma adequada.

In [ ]:
# ETAPA 10 — Enriquecimento da base com atrasos de disponibilidade

PASTA_ETAPA_10 = OUTPUT_DIR / "etapa_10_atrasos"
PASTA_ETAPA_10.mkdir(parents=True, exist_ok=True)

ARQUIVO_BASE_FINAL = (
    INTERIM_DIR / "casos_vsr_menores2.parquet"
)

ARQUIVO_BASE_COM_ATRASO = (
    INTERIM_DIR / "casos_vsr_menores2_com_disponibilidade.parquet"
)

base_final = pd.read_parquet(ARQUIVO_BASE_FINAL)

base_final["nu_notific"] = (
    base_final["nu_notific"]
    .astype("string")
    .str.strip()
)

base_final["dt_sin_pri"] = pd.to_datetime(
    base_final["dt_sin_pri"],
    errors="coerce",
)

CHAVES_JUNCAO = [
    "arquivo_ano",
    "nu_notific",
    "dt_sin_pri",
]

CAMPOS_ATRASO_BRUTO = [
    "NU_NOTIFIC",
    "DT_SIN_PRI",
    "DT_NOTIFIC",
    "DT_DIGITA",
    "DT_PCR",
    "DT_RES_AN",
]

datas_disponibilidade = []

for linha in manifesto_downloads.itertuples(index=False):
    ano = int(linha.ano)

    chaves_ano = base_final.loc[
        base_final["arquivo_ano"] == ano,
        CHAVES_JUNCAO,
    ].copy()

    bruto = pd.read_parquet(
        linha.caminho_local,
        columns=CAMPOS_ATRASO_BRUTO,
    )

    bruto = bruto.rename(
        columns={
            "NU_NOTIFIC": "nu_notific",
            "DT_SIN_PRI": "dt_sin_pri",
            "DT_NOTIFIC": "dt_notific",
            "DT_DIGITA": "dt_digita",
            "DT_PCR": "dt_pcr",
            "DT_RES_AN": "dt_res_an",
        }
    )

    bruto["arquivo_ano"] = ano
    bruto["nu_notific"] = (
        bruto["nu_notific"]
        .astype("string")
        .str.strip()
    )

    for campo in [
        "dt_sin_pri",
        "dt_notific",
        "dt_digita",
        "dt_pcr",
        "dt_res_an",
    ]:
        bruto[campo] = pd.to_datetime(
            bruto[campo],
            errors="coerce",
        )

    # Evita multiplicação acidental de linhas em uma eventual repetição idêntica.
    bruto = bruto.drop_duplicates(
        subset=CHAVES_JUNCAO,
        keep="last",
    )

    datas_ano = chaves_ano.merge(
        bruto,
        on=CHAVES_JUNCAO,
        how="left",
        validate="one_to_one",
    )

    datas_disponibilidade.append(datas_ano)

    del chaves_ano, bruto, datas_ano

datas_disponibilidade = pd.concat(
    datas_disponibilidade,
    ignore_index=True,
)

base_com_atraso = base_final.merge(
    datas_disponibilidade,
    on=CHAVES_JUNCAO,
    how="left",
    validate="one_to_one",
)

# Datas anteriores aos sintomas não podem representar disponibilidade válida.
for campo in [
    "dt_notific",
    "dt_digita",
    "dt_pcr",
    "dt_res_an",
]:
    base_com_atraso.loc[
        base_com_atraso[campo] < base_com_atraso["dt_sin_pri"],
        campo,
    ] = pd.NaT

# Registro disponível: última data válida entre notificação e digitação.
base_com_atraso["dt_disponibilidade_registro"] = (
    base_com_atraso[
        ["dt_notific", "dt_digita"]
    ]
    .max(axis=1)
)

# Resultado de VSR: PCR, antígeno ou a primeira confirmação quando há ambos.
base_com_atraso["dt_resultado_vsr"] = pd.NaT

mascara_pcr = base_com_atraso["vsr_pcr"]
mascara_antigeno = base_com_atraso["vsr_antigeno"]

base_com_atraso.loc[
    mascara_pcr & ~mascara_antigeno,
    "dt_resultado_vsr",
] = base_com_atraso.loc[
    mascara_pcr & ~mascara_antigeno,
    "dt_pcr",
]

base_com_atraso.loc[
    ~mascara_pcr & mascara_antigeno,
    "dt_resultado_vsr",
] = base_com_atraso.loc[
    ~mascara_pcr & mascara_antigeno,
    "dt_res_an",
]

base_com_atraso.loc[
    mascara_pcr & mascara_antigeno,
    "dt_resultado_vsr",
] = base_com_atraso.loc[
    mascara_pcr & mascara_antigeno,
    ["dt_pcr", "dt_res_an"],
].min(axis=1)

# Um caso VSR se torna utilizável quando registro e resultado já são conhecidos.
base_com_atraso["dt_disponibilidade_vsr"] = (
    base_com_atraso[
        ["dt_disponibilidade_registro", "dt_resultado_vsr"]
    ]
    .max(axis=1)
)

base_com_atraso["atraso_registro_dias"] = (
    base_com_atraso["dt_disponibilidade_registro"]
    - base_com_atraso["dt_sin_pri"]
).dt.days

base_com_atraso["atraso_vsr_dias"] = (
    base_com_atraso["dt_disponibilidade_vsr"]
    - base_com_atraso["dt_sin_pri"]
).dt.days

base_com_atraso["atraso_vsr_semanas"] = np.ceil(
    base_com_atraso["atraso_vsr_dias"] / 7
)

base_com_atraso.loc[
    base_com_atraso["atraso_vsr_semanas"] < 0,
    "atraso_vsr_semanas",
] = np.nan

base_com_atraso.to_parquet(
    ARQUIVO_BASE_COM_ATRASO,
    index=False,
)

resumo_atraso = (
    base_com_atraso
    .groupby("arquivo_ano", as_index=False)
    .agg(
        registros=("nu_notific", "size"),
        disponibilidade_registro_ausente=(
            "dt_disponibilidade_registro",
            lambda x: int(x.isna().sum()),
        ),
        resultado_vsr_ausente=(
            "dt_resultado_vsr",
            lambda x: int(x.isna().sum()),
        ),
        disponibilidade_vsr_ausente=(
            "dt_disponibilidade_vsr",
            lambda x: int(x.isna().sum()),
        ),
        atraso_vsr_mediana_dias=(
            "atraso_vsr_dias",
            "median",
        ),
        atraso_vsr_p90_dias=(
            "atraso_vsr_dias",
            lambda x: x.quantile(0.90),
        ),
        atraso_vsr_p95_dias=(
            "atraso_vsr_dias",
            lambda x: x.quantile(0.95),
        ),
    )
)

distribuicao_atraso = (
    base_com_atraso
    .groupby(
        ["arquivo_ano", "atraso_vsr_semanas"],
        dropna=False,
        as_index=False,
    )
    .size()
    .rename(columns={"size": "casos"})
    .sort_values(
        ["arquivo_ano", "atraso_vsr_semanas"],
    )
)

qualidade_enriquecimento = pd.DataFrame(
    [
        {
            "registros_base_final": len(base_final),
            "registros_enriquecidos": len(base_com_atraso),
            "datas_disponibilidade_vsr_ausentes": int(
                base_com_atraso[
                    "dt_disponibilidade_vsr"
                ].isna().sum()
            ),
            "atrasos_negativos": int(
                (base_com_atraso["atraso_vsr_dias"] < 0).sum()
            ),
        }
    ]
)

resumo_atraso.to_csv(
    PASTA_ETAPA_10 / "resumo_atrasos_por_ano.csv",
    index=False,
    encoding="utf-8-sig",
)

distribuicao_atraso.to_csv(
    PASTA_ETAPA_10 / "distribuicao_atraso_semanas.csv",
    index=False,
    encoding="utf-8-sig",
)

qualidade_enriquecimento.to_csv(
    PASTA_ETAPA_10 / "qualidade_enriquecimento_atrasos.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Etapa 10 concluída.")
print(f"Base enriquecida: {ARQUIVO_BASE_COM_ATRASO}")
print(f"Auditorias: {PASTA_ETAPA_10}")

## Etapa 11 — Marcação de maturidade das semanas

Com base nos atrasos observados, uma semana será considerada madura quando tiver pelo menos 13 semanas de distância da data de publicação da fonte.

Semanas recentes serão preservadas, mas não deverão ser usadas como valores finais em treinamento ou avaliação sem ajuste adicional.

In [ ]:
# ETAPA 11 — Marcação de maturidade das semanas

import re

PASTA_ETAPA_11 = OUTPUT_DIR / "etapa_11_maturidade"
PASTA_ETAPA_11.mkdir(parents=True, exist_ok=True)

ARQUIVO_PAINEL_SEMANAL = (
    PROCESSED_DIR / "painel_vsr_semanal.parquet"
)

ARQUIVO_BASE_COM_ATRASO = (
    INTERIM_DIR / "casos_vsr_menores2_com_disponibilidade.parquet"
)

ARQUIVO_PAINEL_MATURO = (
    PROCESSED_DIR / "painel_vsr_semanal_com_maturidade.parquet"
)

JANELA_MATURIDADE_SEMANAS = 13

painel = pd.read_parquet(ARQUIVO_PAINEL_SEMANAL)
base_atraso = pd.read_parquet(ARQUIVO_BASE_COM_ATRASO)

painel["semana"] = pd.to_datetime(painel["semana"])

# Extrai a data de publicação diretamente dos nomes dos arquivos verificados.
def extrair_data_publicacao(nome_arquivo):
    resultado = re.search(
        r"INFLUD\d{2}-(\d{2})-(\d{2})-(\d{4})",
        nome_arquivo,
    )

    if resultado is None:
        return pd.NaT

    dia, mes, ano = resultado.groups()

    return pd.Timestamp(
        year=int(ano),
        month=int(mes),
        day=int(dia),
    )

manifesto_temporal = manifesto_downloads.copy()

manifesto_temporal["data_publicacao"] = (
    manifesto_temporal["arquivo"]
    .apply(extrair_data_publicacao)
)

data_corte_fonte = manifesto_temporal[
    "data_publicacao"
].max()

# Domingo da semana em que ocorreu a publicação mais recente.
semana_corte_fonte = (
    data_corte_fonte.normalize()
    - pd.to_timedelta(
        (data_corte_fonte.dayofweek + 1) % 7,
        unit="D",
    )
)

painel["semanas_ate_corte_fonte"] = (
    (semana_corte_fonte - painel["semana"]).dt.days // 7
).astype(int)

painel["semana_madura_13s"] = (
    painel["cobertura_fechada"]
    & (
        painel["semanas_ate_corte_fonte"]
        >= JANELA_MATURIDADE_SEMANAS
    )
)

# Quantil empírico global do atraso para registrar a justificativa do limiar.
atraso_p95_global = base_atraso[
    "atraso_vsr_dias"
].quantile(0.95)

atraso_p95_global_semanas = int(
    np.ceil(atraso_p95_global / 7)
)

painel.to_parquet(
    ARQUIVO_PAINEL_MATURO,
    index=False,
)

resumo_maturidade = (
    painel
    .groupby(
        ["nivel", "semana_madura_13s"],
        dropna=False,
        as_index=False,
    )
    .agg(
        linhas=("nivel", "size"),
        unidades=("unidade", "nunique"),
        casos_vsr=("casos_vsr", "sum"),
    )
)

calendario_maturidade = (
    painel[
        painel["nivel"] == "BRASIL"
    ][
        [
            "semana",
            "ano_epi",
            "semana_epi",
            "cobertura_fechada",
            "semanas_ate_corte_fonte",
            "semana_madura_13s",
            "casos_vsr",
        ]
    ]
    .sort_values("semana")
    .reset_index(drop=True)
)

validacao_maturidade = pd.DataFrame(
    [
        {
            "data_corte_fonte": data_corte_fonte,
            "semana_corte_fonte": semana_corte_fonte,
            "janela_maturidade_semanas": JANELA_MATURIDADE_SEMANAS,
            "atraso_p95_global_dias": atraso_p95_global,
            "atraso_p95_global_semanas": atraso_p95_global_semanas,
            "primeira_semana_madura_ausente": (
                calendario_maturidade.loc[
                    ~calendario_maturidade[
                        "semana_madura_13s"
                    ],
                    "semana",
                ].min()
            ),
            "ultima_semana_madura": (
                calendario_maturidade.loc[
                    calendario_maturidade[
                        "semana_madura_13s"
                    ],
                    "semana",
                ].max()
            ),
        }
    ]
)

resumo_maturidade.to_csv(
    PASTA_ETAPA_11 / "resumo_maturidade_painel.csv",
    index=False,
    encoding="utf-8-sig",
)

calendario_maturidade.to_csv(
    PASTA_ETAPA_11 / "calendario_maturidade_brasil.csv",
    index=False,
    encoding="utf-8-sig",
)

validacao_maturidade.to_csv(
    PASTA_ETAPA_11 / "validacao_limiar_maturidade.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Etapa 11 concluída.")
print(f"Painel com maturidade: {ARQUIVO_PAINEL_MATURO}")
print(f"Data de corte da fonte: {data_corte_fonte.date()}")
print(f"Janela de maturidade: {JANELA_MATURIDADE_SEMANAS} semanas")
print(
    f"P95 global do atraso: "
    f"{atraso_p95_global:.1f} dias "
    f"({atraso_p95_global_semanas} semanas)"
)
print(f"Resultados: {PASTA_ETAPA_11}")